This notebook turns the synthetic handwriting images made by the `synthetic` package into training samples. It runs Google's [InkSight](https://github.com/google-research/inksight) Small-p model on every word of every image in `data/synthetic/labels.json`, then converts each line's ink into `[dx, dy, pen_down, pen_up, eos]` rows, exactly as `03-combine-data.ipynb` writes them.

It is `06-inksight-stroke-generation.ipynb` turned into a loop over the whole synthetic dataset, with one simplification: **no OCR**. The renderer saves the box around every word it draws, so each word is cut out exactly. And since we know what each word says, the model is prompted with `Derender the ink: <word>`, the mode InkSight does best in.

Use a GPU (on Colab, a T4 runtime). Inference is batched: each call to the model derenders `BATCH_SIZE` words at once. The model's output is cached as it goes, so a run that is interrupted carries on where it stopped.

# Setup

On Colab, the next cell installs the versions of TensorFlow and TensorFlow Text that the InkSight checkpoint works with. Restart the session afterwards (*Runtime ▸ Restart session*) and run the notebook from the top. Locally, `uv sync` installs everything and the cell does nothing.

In [1]:
import sys

if "google.colab" in sys.modules:
    %pip install --quiet "tensorflow[and-cuda]==2.20.0" "tensorflow-text==2.20.1"

In [2]:
# The released InkSight checkpoint needs the XLA matrix multiplication that TensorFlow used
# before version 2.18 (see the InkSight repo's colab.ipynb). The flags only matter on a GPU,
# and must be set before TensorFlow is imported.

import os

INKSIGHT_XLA_FLAGS = (
    "--xla_gpu_autotune_level=0 "
    "--xla_disable_hlo_passes=custom-kernel-fusion-rewriter,custom_kernel-fusion-autotuner"
)

if INKSIGHT_XLA_FLAGS not in os.environ.get("XLA_FLAGS", ""):
    os.environ["XLA_FLAGS"] = f"{os.environ.get('XLA_FLAGS', '')} {INKSIGHT_XLA_FLAGS}".strip()

In [3]:
from datetime import datetime
import json
from pathlib import Path
import re
import time
from typing import NamedTuple

from huggingface_hub import snapshot_download
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from scipy.stats import theilslopes
import tensorflow as tf
import tensorflow_text  # registers the text operations the InkSight model uses

print("TensorFlow", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU") or "none (inference will be slow)")

TensorFlow 2.21.0
GPUs: none (inference will be slow)


# Load Data

Your project should be structured as follows:

```
.
├── data/
│   ├── synthetic/
│   │   ├── images/
│   │   └── labels.json
│   └── cleaned/
└── notebooks/
```

The images and `labels.json` come from `uv run python -m synthetic.render_images`. On Colab, mount Google Drive and point `ROOT` at the project there.

In [4]:
# Change this to your project root. On Colab, mount Google Drive first:
#
#     from google.colab import drive
#     drive.mount("/content/drive")
#     ROOT = Path("/content/drive/MyDrive/handwriting-synthesis")

ROOT = Path("/Users/jacob/github.com/jda5/handwriting-synthesis")

SYNTHETIC_DIR = ROOT / "data" / "synthetic"
if not SYNTHETIC_DIR.exists():
    raise FileNotFoundError("the 'synthetic' subdirectory does not exist")

CLEANED_DIR = ROOT / "data" / "cleaned"
if not CLEANED_DIR.exists():
    raise FileNotFoundError("the 'cleaned' subdirectory does not exist")

Each label names an image and holds its text, the font it was drawn in, and `word_boxes`: the box around each word's ink, as `[left, top, right, bottom]` pixels, in the order `text.split()` gives the words.

In [5]:
with open(SYNTHETIC_DIR / "labels.json", encoding="utf-8") as f:
    labels = json.load(f)

print("Images:", len(labels))
print("Words:", sum(len(label["word_boxes"]) for label in labels))

labels[0]

Images: 1643
Words: 7003


{'image': 'postcodes-0000.png',
 'text': 'DL14 8PQ',
 'source': 'postcodes',
 'font': 'David Reid CAP',
 'word_boxes': [[20, 29, 185, 85], [216, 20, 344, 83]]}

# InkSight

InkSight reads a 224 × 224 image and a text prompt, and writes the ink it sees as text. Each word is cut out of its line, scaled to fit the 224 × 224 square, and centred on black, as the model expects.

Throughout this notebook, **ink** is a list of strokes, and each stroke is an `(N, 2)` array of the `x, y` pixel coordinates the pen passes through while it touches the paper. `y` grows downwards, as in an image.

In [6]:
# Download the InkSight Small-p model from Hugging Face. The loaded model is kept in a variable
# of its own, so its weights stay alive while its serving function is in use.

inksight = tf.saved_model.load(snapshot_download("Derendering/InkSight-Small-p"))
derender_fn = inksight.signatures["serving_default"]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

In [7]:
IMAGE_SIZE = 224  # InkSight reads and writes 224 x 224 images


# A word cut out of its line image and prepared for InkSight. `content` is the part of the
# 224 x 224 image the word fills; the rest is black padding.

class WordCrop(NamedTuple):
    image: Image.Image
    box: tuple[int, int, int, int]  # where the word was cut from, in line image pixels
    ratio: float  # how much the word was scaled to fit
    content: tuple[int, int, int, int]  # left, top, right, bottom in the 224 x 224 image

In [8]:
# Cuts a word out of its line image, with a margin of `pad` times its size on every side so
# its ascenders and descenders are not clipped. Then scales it to fit the 224 x 224 square,
# keeping its aspect ratio, and centres it on black.

def crop_word(image: Image.Image, box: list[int], pad: float = 0.1) -> WordCrop:
    x0, y0, x1, y1 = box
    px, py = (x1 - x0) * pad, (y1 - y0) * pad
    box = (
        int(max(0, x0 - px)),
        int(max(0, y0 - py)),
        int(min(image.width, x1 + px)),
        int(min(image.height, y1 + py)),
    )
    word = image.crop(box)

    ratio = min(IMAGE_SIZE / word.width, IMAGE_SIZE / word.height)
    word = word.resize((max(1, int(word.width * ratio)), max(1, int(word.height * ratio))))

    square = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (0, 0, 0))
    left = (IMAGE_SIZE - word.width) // 2
    top = (IMAGE_SIZE - word.height) // 2
    square.paste(word, (left, top))

    return WordCrop(square, box, ratio, (left, top, left + word.width, top + word.height))

In [9]:
# Moves a stroke from the 224 x 224 image back onto the line image it was cut from

def to_line(stroke: np.ndarray, crop: WordCrop) -> np.ndarray:
    left, top = crop.content[:2]
    return (stroke - (left, top)) / crop.ratio + crop.box[:2]

The model writes ink as tokens: `<ink_token_450>` starts a new stroke, and is followed by pairs of tokens, an `x` token (0 to 224) then a `y` token (225 to 449, which is `y + 225`). Anything that does not fit this pattern is skipped, as in the InkSight notebook.

In [10]:
STROKE_START = 2 * (IMAGE_SIZE + 1)  # 450
Y_OFFSET = IMAGE_SIZE + 1  # 225


# Reads the ink token numbers out of the model's output text

def read_tokens(output: str) -> list[int]:
    return [int(token) for token in re.findall(r"<ink_token_(\d+)>", output)]


# Turns ink tokens into ink, in the 224 x 224 image's coordinates

def tokens_to_ink(tokens: list[int]) -> list[np.ndarray]:
    ink, stroke = [], []
    i = 0

    while i < len(tokens):
        if tokens[i] == STROKE_START:
            if stroke:
                ink.append(np.array(stroke, dtype=float))
            stroke = []
            i += 1

        elif i + 1 < len(tokens) and tokens[i + 1] != STROKE_START:
            x, y = tokens[i], tokens[i + 1] - Y_OFFSET
            if 0 <= x <= IMAGE_SIZE and 0 <= y <= IMAGE_SIZE:
                stroke.append((x, y))
            i += 2

        else:
            i += 1  # an x with no y

    if stroke:
        ink.append(np.array(stroke, dtype=float))

    return ink

### Inference

The model takes a batch of JPEG-encoded images and a batch of prompts, and derenders them all in one call. A word that comes back without any ink is tried again with the plain `Derender the ink.` prompt, as the InkSight notebook does.

In [11]:
BATCH_SIZE = 16  # words per call to the model; lower it if the GPU runs out of memory


# Runs InkSight on images with their prompts, BATCH_SIZE at a time, and returns each image's
# ink tokens

def run_inksight(images: list[Image.Image], prompts: list[str]) -> list[list[int]]:
    tokens = []

    for start in range(0, len(images), BATCH_SIZE):
        batch_images = images[start : start + BATCH_SIZE]
        batch_prompts = prompts[start : start + BATCH_SIZE]

        encoded = [tf.io.encode_jpeg(np.array(image)) for image in batch_images]
        output = derender_fn(
            **{
                "input_text": tf.constant(batch_prompts, dtype=tf.string),
                "image/encoded": tf.reshape(tf.stack(encoded), (len(encoded), 1)),
            }
        )
        tokens += [read_tokens(text.decode()) for text in output["output_0"].numpy()[:, 0]]

    return tokens

In [12]:
# Derenders word crops, prompting the model with the words they show

def derender(crops: list[WordCrop], words: list[str]) -> list[list[int]]:
    images = [crop.image for crop in crops]
    tokens = run_inksight(images, [f"Derender the ink: {word}" for word in words])

    empty = [i for i, word_tokens in enumerate(tokens) if not tokens_to_ink(word_tokens)]
    if empty:
        retried = run_inksight([images[i] for i in empty], ["Derender the ink."] * len(empty))
        for i, word_tokens in zip(empty, retried):
            tokens[i] = word_tokens

    return tokens

### Derender Every Word

The images are derendered a chunk at a time. After each chunk, the model's output (the ink tokens of each image's words) is saved to `inksight-tokens.json`, and images already in that file are skipped. So if the run stops, running this cell again carries on from where it was.

If you re-render the images, delete `inksight-tokens.json` first, or the old ink will be reused.

In [13]:
TOKENS_PATH = SYNTHETIC_DIR / "inksight-tokens.json"
IMAGES_PER_CHUNK = 64

# image name -> one list of ink tokens per word
if TOKENS_PATH.exists():
    with open(TOKENS_PATH) as f:
        image_tokens = json.load(f)
else:
    image_tokens = {}

remaining = [label for label in labels if label["image"] not in image_tokens]
print(f"{len(image_tokens)} images already derendered, {len(remaining)} to go")

0 images already derendered, 1643 to go


In [ ]:
started = time.perf_counter()

for start in range(0, len(remaining), IMAGES_PER_CHUNK):
    chunk = remaining[start : start + IMAGES_PER_CHUNK]

    # Cut out every word in the chunk, remembering how many words each image has
    crops, words = [], []
    for label in chunk:
        image = Image.open(SYNTHETIC_DIR / "images" / label["image"]).convert("RGB")
        for word, box in zip(label["text"].split(), label["word_boxes"]):
            crops.append(crop_word(image, box))
            words.append(word)

    tokens = derender(crops, words)

    # Hand each image its own words' tokens, in order
    for label in chunk:
        count = len(label["word_boxes"])
        image_tokens[label["image"]], tokens = tokens[:count], tokens[count:]

    with open(TOKENS_PATH, "w") as f:
        json.dump(image_tokens, f, separators=(",", ":"))

    done = start + len(chunk)
    minutes = (time.perf_counter() - started) / 60
    print(f"{done}/{len(remaining)} images in {minutes:.1f} min")

I0000 00:00:1793706625.904203 8933835 service.cc:153] XLA service 0xac0244500 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1793706625.904214 8933835 service.cc:161]   StreamExecutor [0]: Host, Default Version (Driver: 0.0.0; Runtime: 0.0.0; Toolkit: 0.0.0; DNN: 0.0.0)
I0000 00:00:1793706626.129519 8933835 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
W0000 00:00:1793706626.226075 8933835 assert_op.cc:39] Ignoring Assert operator jax2tf_unknown_/assert_equal_1/Assert/AssertGuard/Assert
W0000 00:00:1793706626.526145 8933835 assert_op.cc:39] Ignoring Assert operator jax2tf_unknown_/assert_equal_3/Assert/AssertGuard/Assert
W0000 00:00:1793706626.526561 8933835 assert_op.cc:39] Ignoring Assert operator jax2tf_unknown_/assert_equal_5/Assert/AssertGuard/Assert
W0000 00:00:1793706626.527952 8933835 assert_op.cc:39] Ignoring Assert operator jax2tf_unknown_/assert_equal_7/Ass

# Clean the Ink

The model sometimes adds marks that are not in the image:

1. A single point at the very edge of its 224 × 224 input, where a wide word's image meets the side of the square.
2. Small stray dots inside the word.

A stroke counts as a mark when it is shorter than `MAX_MARK_LENGTH` times the height of the word's ink. Marks at the edge are dropped. Then each word keeps only as many of its remaining small marks as its text has characters drawn with a detached dot or tick (i, j and some punctuation), choosing the marks closest to the rest of the ink.

In [ ]:
MAX_MARK_LENGTH = 0.12  # as a fraction of the word's ink height
EDGE = 4  # pixels of the 224 x 224 image

DOT_CHARACTERS = set("ij.:;!?'\"")  # characters whose glyph includes a detached dot or tick


def stroke_length(stroke: np.ndarray) -> float:
    return float(np.hypot(*np.diff(stroke, axis=0).T).sum())


def ink_height(ink: list[np.ndarray]) -> float:
    y = np.concatenate(ink)[:, 1]
    return float(y.max() - y.min()) or 1.0

In [ ]:
# Drops the small marks lying along one edge of `content`, the part of the 224 x 224 image the
# word fills. Works in the 224 x 224 image's coordinates.

def remove_edge_marks(ink: list[np.ndarray], content: tuple[int, int, int, int]) -> list[np.ndarray]:
    if len(ink) < 2:
        return ink

    x0, y0, x1, y1 = content
    max_length = MAX_MARK_LENGTH * ink_height(ink)

    kept = []
    for stroke in ink:
        x, y = stroke.T
        at_edge = (
            (x <= x0 + EDGE).all()
            or (x >= x1 - EDGE).all()
            or (y <= y0 + EDGE).all()
            or (y >= y1 - EDGE).all()
        )
        if not (at_edge and stroke_length(stroke) < max_length):
            kept.append(stroke)

    return kept or ink

In [ ]:
# Keeps as many small marks as `word` has characters with a detached dot, choosing the
# marks nearest the rest of the ink, and drops the others

def remove_stray_dots(ink: list[np.ndarray], word: str) -> list[np.ndarray]:
    if len(ink) < 2:
        return ink

    max_length = MAX_MARK_LENGTH * ink_height(ink)
    marks = [i for i, stroke in enumerate(ink) if stroke_length(stroke) < max_length]
    rest = [i for i in range(len(ink)) if i not in marks]
    expected = sum(char in DOT_CHARACTERS for char in word)

    if len(marks) <= expected or not rest:
        return ink

    rest_points = np.concatenate([ink[i] for i in rest])

    def distance_to_rest(i: int) -> float:
        offsets = ink[i][:, None, :] - rest_points[None, :, :]
        return np.hypot(offsets[..., 0], offsets[..., 1]).min()

    kept_marks = sorted(marks, key=distance_to_rest)[:expected]
    return [stroke for i, stroke in enumerate(ink) if i not in marks or i in kept_marks]

In [ ]:
# Rebuilds a line's ink from its words' tokens: each word's ink is cleaned, then moved back
# onto the line image, and the words are joined in reading order

def line_ink(label: dict, tokens: list[list[int]]) -> list[np.ndarray]:
    image = Image.open(SYNTHETIC_DIR / "images" / label["image"]).convert("RGB")
    ink = []

    for word, box, word_tokens in zip(label["text"].split(), label["word_boxes"], tokens):
        crop = crop_word(image, box)
        word_ink = tokens_to_ink(word_tokens)
        word_ink = remove_edge_marks(word_ink, crop.content)
        word_ink = remove_stray_dots(word_ink, word)
        ink += [to_line(stroke, crop) for stroke in word_ink]

    return ink

In [ ]:
# Draws a line's ink over its image, to check the two line up

def plot_line(label: dict, ink: list[np.ndarray]):
    image = Image.open(SYNTHETIC_DIR / "images" / label["image"])

    plt.figure(figsize=(12, 3))
    plt.imshow(image, cmap="gray", alpha=0.4)
    for stroke in ink:
        plt.plot(stroke[:, 0], stroke[:, 1], ".-", color="crimson", markersize=2, linewidth=1)

    plt.title(f"{label['text']}  ({label['font']})")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
for label in labels[:: len(labels) // 5][:5]:
    plot_line(label, line_ink(label, image_tokens[label["image"]]))

# From Ink to Samples

The training data is a list of float32 arrays of shape `(N, 5)` with rows `[dx, dy, pen_down, pen_up, eos]`. The first row is the start token `[0, 0, 1, 0, 0]`, the last is the end token `[0, 0, 0, 0, 1]`, and the point at which the pen lifts carries `pen_up = 1`. In `03-combine-data.ipynb` every sample is shifted, slant-aligned, and divided by its dataset's median *handwriting scale* (the interquartile range of the points' heights about a Theil–Sen baseline), so one unit is roughly the height of the main body of the letters.

InkSight's ink goes through the same steps, with two differences:

- **Scale** is taken from each line itself, so every sample has a handwriting scale of exactly 1. The training samples sit between about 0.8 and 1.25.
- **Resampling.** InkSight spaces points by shape, while the training data was sampled in time (about 20 points per character, a median of 0.19 units between consecutive pen-down points). Each stroke is re-spaced along its length to `TARGET_STEP` units, so the `dx, dy` the model learns look alike.

Every stroke's last point becomes a pen lift. A single-point stroke (an i-dot or a full stop) becomes a pen-down and a pen-up at the same spot, as the DeepWriting export does.

In [ ]:
# Distance between consecutive pen-down points, in handwriting scale units (training data:
# median 0.19, interquartile range 0.11 to 0.30). Set it to 0 to keep InkSight's own points.

TARGET_STEP = 0.19

The next few cells are copied from `03-combine-data.ipynb`.

In [ ]:
class Point(NamedTuple):
    x: float
    y: float
    pen_up: bool


# End of sequence token that models "finished writing"

eos_token = (0, 0, 0, 0, 1)

In [ ]:
# A function that turns a list of Points to a list of strokes (dx, dy offsets)
# as well as a one-hot encoding of the pen state

def calculate_strokes(points: list[Point], include_eos: bool = True) -> np.ndarray:
    strokes = []
    prev_x, prev_y = None, None

    for point in points:
        dx = 0 if prev_x is None else point.x - prev_x
        dy = 0 if prev_y is None else point.y - prev_y
        strokes.append((dx, dy, not point.pen_up, point.pen_up, 0))
        prev_x, prev_y = point.x, point.y

    if include_eos:
        strokes.append(eos_token)

    return np.array(strokes, dtype=np.float32)

In [ ]:
# Shifts the points so the leftmost sits at x = 0 and the topmost at y = 0

def shift_points(points: list[Point]) -> list[Point]:
    min_x = min(p.x for p in points)
    min_y = min(p.y for p in points)
    return [Point(p.x - min_x, p.y - min_y, p.pen_up) for p in points]

In [ ]:
# Fits a Theil-Sen baseline through the centres of the strokes

def fit_baseline(coords: list[np.ndarray]) -> tuple[float, float]:
    centres = np.array([np.median(c, axis=0) for c in coords])
    if len(centres) > 2:
        slope, offset = theilslopes(centres[:, 1], centres[:, 0])[:2]
    else:
        slope, offset = 0.0, np.median(centres[:, 1])
    return slope, offset


# The interquartile range of the points' heights about the baseline: roughly the height of
# the main body of the letters

def handwriting_scale(points: list[Point]) -> float:
    strokes, current = [], []
    for point in points:
        current.append(point)
        if point.pen_up:
            strokes.append(np.array([(p.x, p.y) for p in current]))
            current = []
    if current:
        strokes.append(np.array([(p.x, p.y) for p in current]))

    slope, offset = fit_baseline(strokes)
    heights = np.concatenate([c[:, 1] - (slope * c[:, 0] + offset) for c in strokes])
    q1, q3 = np.percentile(heights, [25, 75])
    return q3 - q1

In [ ]:
# Rotates the points so a least-squares line through them is horizontal

def _align(xy: np.ndarray) -> np.ndarray:
    xy = np.copy(xy)
    x, y = xy[:, 0].reshape(-1, 1), xy[:, 1].reshape(-1, 1)
    design = np.concatenate([np.ones([x.shape[0], 1]), x], axis=1)
    offset, slope = np.linalg.inv(design.T.dot(design)).dot(design.T).dot(y).squeeze()
    theta = np.arctan(slope)
    rotation = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    return np.dot(xy, rotation) - offset


def align_points(points: list[Point]) -> list[Point]:
    point_array = np.array([[point.x, point.y] for point in points], dtype=np.float32)
    aligned_points = _align(point_array)
    return [
        Point(x=float(a[0]), y=float(a[1]), pen_up=point.pen_up)
        for a, point in zip(aligned_points, points)
    ]

The rest are new.

In [ ]:
# Turns ink into Points. The last point of every stroke carries the pen lift, and a single
# point stroke (a dot) becomes a pen-down and a pen-up at the same spot.

def ink_to_points(ink: list[np.ndarray]) -> list[Point]:
    points = []
    for stroke in ink:
        if len(stroke) == 1:
            stroke = np.repeat(stroke, 2, axis=0)
        points += [Point(float(x), float(y), False) for x, y in stroke[:-1]]
        points.append(Point(float(stroke[-1, 0]), float(stroke[-1, 1]), True))
    return points

In [ ]:
# Re-spaces every stroke so consecutive points are `step` apart along it, keeping the first and
# last point of each stroke. The jumps between strokes, with the pen lifted, are left alone.

def resample_points(points: list[Point], step: float) -> list[Point]:
    strokes, current = [], []
    for point in points:
        current.append(point)
        if point.pen_up:
            strokes.append(current)
            current = []
    if current:
        strokes.append(current)

    resampled = []
    for stroke in strokes:
        xy = np.array([(p.x, p.y) for p in stroke], dtype=float)

        # Drop repeated points, so the distance along the stroke always increases
        if len(xy) > 1:
            xy = xy[np.concatenate([[True], np.hypot(*np.diff(xy, axis=0).T) > 0])]

        if len(xy) < 2:
            new = np.repeat(xy, 2, axis=0)  # a dot: pen down, pen up
        else:
            distance = np.concatenate([[0.0], np.cumsum(np.hypot(*np.diff(xy, axis=0).T))])
            count = max(2, int(round(distance[-1] / step)) + 1)
            targets = np.linspace(0, distance[-1], count)
            new = np.column_stack(
                [np.interp(targets, distance, xy[:, 0]), np.interp(targets, distance, xy[:, 1])]
            )

        resampled += [Point(float(x), float(y), False) for x, y in new[:-1]]
        resampled.append(Point(float(new[-1, 0]), float(new[-1, 1]), True))

    return resampled

In [ ]:
# Turns a line's ink (in image pixels) into an (N, 5) sample in the training data's units

def ink_to_sample(ink: list[np.ndarray]) -> np.ndarray:
    points = align_points(shift_points(ink_to_points(ink)))

    pixel_scale = handwriting_scale(points)
    if not pixel_scale > 0:
        # e.g. one flat stroke, with no spread in height: fall back to the bounding box
        ys = [p.y for p in points]
        pixel_scale = max(max(ys) - min(ys), 1.0)

    if TARGET_STEP > 0:
        points = resample_points(points, TARGET_STEP * pixel_scale)
        pixel_scale = handwriting_scale(points) or pixel_scale

    points = shift_points([Point(p.x / pixel_scale, p.y / pixel_scale, p.pen_up) for p in points])
    return calculate_strokes(points)

In [ ]:
samples, texts = [], []

for label in labels:
    ink = line_ink(label, image_tokens[label["image"]])
    if not ink:
        print(f"Skipped {label['image']}: no ink")
        continue

    samples.append(ink_to_sample(ink))
    texts.append(label["text"])

points_per_char = np.median([(len(s) - 2) / len(t) for s, t in zip(samples, texts)])

print("Samples:", len(samples))
print(f"Median points per character: {points_per_char:.1f} (training data: about 20)")

### Visualise

Drawn as in `04-model-improved.ipynb`: the yellow dotted line is the pen's whole path, so the jumps with the pen lifted show as faint gaps between the black strokes.

In [ ]:
def draw_strokes(strokes: np.ndarray, label: str | None = None):
    coordinates = strokes[:, :2].cumsum(axis=0)
    pen_lifts = np.flatnonzero(strokes[:, 3])

    plt.figure(figsize=(12, 3))
    plt.plot(coordinates[:, 0], -coordinates[:, 1], "y:", alpha=0.4)

    for stroke in np.split(coordinates, pen_lifts + 1)[:-1]:
        plt.plot(stroke[:, 0], -stroke[:, 1], ".-", color="black")

    plt.title(label)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
for i in range(0, len(samples), len(samples) // 5)[:5]:
    draw_strokes(samples[i], f"{texts[i]}  ({len(samples[i])} steps)")

# Save

The samples are saved to `data/cleaned/synthetic-<timestamp>/` in the layout `04-model-improved.ipynb` loads, after the checks it makes when it loads them.

In [ ]:
sos = np.array([0, 0, 1, 0, 0], dtype=np.float32)
eos = np.array(eos_token, dtype=np.float32)

assert len(samples) == len(texts)
assert all((s[0] == sos).all() and (s[-1] == eos).all() for s in samples)
assert all(np.isfinite(s).all() for s in samples)

In [ ]:
SAVE_DIR = CLEANED_DIR / ("synthetic-" + datetime.now().strftime("%Y-%m-%dT%H-%M-%S"))
SAVE_DIR.mkdir()

np.savez(SAVE_DIR / "strokes_data.npz", *samples)

with open(SAVE_DIR / "texts.json", "w", encoding="utf-8") as f:
    json.dump(texts, f, ensure_ascii=False, separators=(",", ":"))

print(f"Saved {len(samples)} samples to {SAVE_DIR}")

`04-model-improved.ipynb` trains on the newest `combined-*` folder, so to train on these samples, append them to the latest combined set:

```python
def load(path):
    with np.load(path / "strokes_data.npz") as data:
        strokes = [data[f"arr_{i}"] for i in range(len(data.files))]
    with open(path / "texts.json") as f:
        return strokes, json.load(f)


base_strokes, base_texts = load(max(CLEANED_DIR.glob("combined-*")))
new_strokes, new_texts = load(max(CLEANED_DIR.glob("synthetic-*")))

out = CLEANED_DIR / f"combined-{datetime.now().strftime('%Y-%m-%dT%H-%M-%S')}"
out.mkdir()
np.savez(out / "strokes_data.npz", *base_strokes, *new_strokes)
with open(out / "texts.json", "w", encoding="utf-8") as f:
    json.dump(base_texts + new_texts, f, ensure_ascii=False, separators=(",", ":"))
```

Things to keep in mind:

- These samples are InkSight's guess at how the writing was produced, not recorded pen data. Stroke order and direction are usually plausible but not guaranteed, so look through the plots above.
- The resampled points are evenly spaced, whereas real pen data slows down in curves and speeds up on straight runs.
- The new characters (`$`, `@`, `£`, `%`) extend the vocabulary that `TextVectorization` adapts in `04-model-improved.ipynb`, so the model has to be trained afresh rather than fine-tuned from a checkpoint whose `config.json` fixed the old vocabulary.